In [1]:
import requests
import json
import numpy as np
import pandas as pd
import time
import warnings
import random
from collections import Counter as Counter

In [2]:
pd.set_option('display.max_rows', 5000)

# Collecting Fields Based on 2024 Variable Names

I identified the variable names based on their names in the 2024 ACS Detailed Tables Variables and 2024 ACS Data Profiles Variables tables (found [here](https://api.census.gov/data/2024/acs/acs5/variables.html) and [here](https://api.census.gov/data/2024/acs/acs5/profile/variables.html), respectively), using this notebook to validate the variable names

I later found that the variable names can differ from year to year. I updated this notebook to scrape the JSON versions of these tables across the various years, finding the variable names based on their Label in each table

In [3]:
demographic_fields = ['DP05_0002E', 'DP05_0003E'] + \
    [f'DP05_000{str(i)}E' for i in range(5, 10)] + \
    [f'DP05_00{str(i)}E' for i in range(10, 18)] + \
    [f'DP05_00{str(i)}E' for i in [35, 45, 53, 61, 69, 74, 90, 96]]

commuting_fields = ['B08301_003E', 'B08301_004E'] + \
    [f'B08301_0{i}E' for i in [10, 16, 17, 18, 19, 20, 21]] + \
    [f'B08006_0{i}E' for i in [20, 21, 25, 31, 32, 33, 34, 37, 38, 42, 48, 49, 50, 51]]

commuting_by_age_fields = [f'B08101_0{i}E' for i in list(range(10, 17)) + list(range(18, 25)) + 
    list(range(26, 33)) + list(range(34, 41)) + list(range(42, 49)) + list(range(50, 57))]

commuting_by_race_fields = [f'B08105B_00{i}E' for i in range(2, 8)] + \
    [f'B08105C_00{i}E' for i in range(2, 8)] + \
    [f'B08105D_00{i}E' for i in range(2, 8)] + \
    [f'B08105E_00{i}E' for i in range(2, 8)] + \
    [f'B08105F_00{i}E' for i in range(2, 8)] + \
    [f'B08105G_00{i}E' for i in range(2, 8)] + \
    [f'B08105H_00{i}E' for i in range(2, 8)] + \
    [f'B08105I_00{i}E' for i in range(2, 8)]

# Validating the Column Names

In [4]:
demographic_fields_df = pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Demographics.csv')
demographic_fields_df = demographic_fields_df[(demographic_fields_df.Name.str.contains('DP0'))]
commuting_fields_df = pd.concat([pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Commuting.csv'), 
    pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Commuting by Sex.csv')])
commuting_by_age_fields_df = pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Commuting by Age.csv')
commuting_by_race_fields_df = pd.read_csv(
    'Data/Reference/US Census Bureau Data Documentation - Commuting by Race_Ethnicity.csv')

In [5]:
demographic_fields_df = pd.read_csv('Data/Reference/US Census Bureau Data Documentation - Demographics.csv')
demographic_fields_df[~(demographic_fields_df.Name.str.contains('DP0'))]

,Name,Label,Concept,Required,Attributes,Limit,Predicate Type,Group
864,GEO_ID,Geography,Selected Social Characteristics in the United ...,predicate-only,NaN,0,string,DP05
865,NAME,Geographic Area Name,Selected Social Characteristics in the United ...,predicate-only,NaN,0,string,DP05
1403,for,Census API FIPS 'for' clause,Census API Geography Specification,predicate-only,NaN,0,fips-for,NaN
1404,GEO_ID,Geography,Selected Social Characteristics in the United ...,not required,NAME,0,string,"DP04, DP05, DP02PR, DP02, DP03"
1405,GEOCOMP,GEO_ID Component,NaN,default displayed,NaN,0,string,NaN
1406,in,Census API FIPS 'in' clause,Census API Geography Specification,predicate-only,NaN,0,fips-in,NaN
1407,METDIV,Metropolitan Division,NaN,not required,NaN,0,(not a predicate),NaN
1408,NATION,Nation,NaN,not required,NaN,0,(not a predicate),NaN
1409,PLACE,Place,NaN,not required,NaN,0,(not a predicate),NaN
1410,PRINCITY,Principal City,NaN,not required,NaN,0,(not a predicate),NaN


In [6]:
[chars for chars in 'asdf'.split('!!')]

['asdf']

In [7]:
def clean_fields_df(fields_df):
    fields_df['LabelDepth'] = [label.count('!!') for label in fields_df.Label]

    fields_df['Metric'] = [label.split('!!')[0] for label in fields_df.Label]
    fields_df['DemographicLabel'] = [label.split('!!')[1] if label.count('!!') >= 1 else None
                                         for label in fields_df.Label]
    fields_df['SubLabel1'] = [label.split('!!')[2] if label.count('!!') >= 2 else 
                              None for label in fields_df.Label]
    fields_df['SubLabel2'] = [label.split('!!')[3] if label.count('!!') >= 3 else 
                              None for label in fields_df.Label]
    fields_df['SubLabel3'] = [label.split('!!')[4] if label.count('!!') >= 4 else 
                              None for label in fields_df.Label]
    fields_df['SubLabel4'] = [label.split('!!')[5] if label.count('!!') >= 5 else 
                              None for label in fields_df.Label]
    fields_df['FinalSubLabel'] = [label.split('!!')[label.count('!!')] for label in fields_df.Label]
    
    return(fields_df)

demographic_fields_df = clean_fields_df(demographic_fields_df)
commuting_fields_df = clean_fields_df(commuting_fields_df)
commuting_by_age_fields_df = clean_fields_df(commuting_by_age_fields_df)
commuting_by_race_fields_df = clean_fields_df(commuting_by_race_fields_df)

In [8]:
all_field_names_df = pd.concat([demographic_fields_df, commuting_fields_df, 
                                commuting_by_age_fields_df, commuting_by_race_fields_df])
all_field_names_df['FieldName'] = all_field_names_df.SubLabel3.str.cat(all_field_names_df.SubLabel4)

In [9]:
all_field_names_df.loc[(all_field_names_df.Name.isin(
    demographic_fields + commuting_fields + commuting_by_age_fields + commuting_by_race_fields
))][['Name', 'Label', 'SubLabel1', 'SubLabel2', 'SubLabel3', 'SubLabel4', 'FinalSubLabel']]

,Name,Label,SubLabel1,SubLabel2,SubLabel3,SubLabel4,FinalSubLabel
8,DP05_0002E,Estimate!!SEX AND AGE!!Total population!!Male,Total population,Male,None,None,Male
16,DP05_0003E,Estimate!!SEX AND AGE!!Total population!!Female,Total population,Female,None,None,Female
32,DP05_0005E,Estimate!!SEX AND AGE!!Total population!!Under...,Total population,Under 5 years,None,None,Under 5 years
40,DP05_0006E,Estimate!!SEX AND AGE!!Total population!!5 to ...,Total population,5 to 9 years,None,None,5 to 9 years
48,DP05_0007E,Estimate!!SEX AND AGE!!Total population!!10 to...,Total population,10 to 14 years,None,None,10 to 14 years
56,DP05_0008E,Estimate!!SEX AND AGE!!Total population!!15 to...,Total population,15 to 19 years,None,None,15 to 19 years
64,DP05_0009E,Estimate!!SEX AND AGE!!Total population!!20 to...,Total population,20 to 24 years,None,None,20 to 24 years
72,DP05_0010E,Estimate!!SEX AND AGE!!Total population!!25 to...,Total population,25 to 34 years,None,None,25 to 34 years
80,DP05_0011E,Estimate!!SEX AND AGE!!Total population!!35 to...,Total population,35 to 44 years,None,None,35 to 44 years
88,DP05_0012E,Estimate!!SEX AND AGE!!Total population!!45 to...,Total population,45 to 54 years,None,None,45 to 54 years


In [10]:
all_field_names_df.loc[(all_field_names_df.Name.isin(
    demographic_fields + commuting_fields + commuting_by_age_fields + commuting_by_race_fields
))].Name.unique().tolist()

['DP05_0002E',
 'DP05_0003E',
 'DP05_0005E',
 'DP05_0006E',
 'DP05_0007E',
 'DP05_0008E',
 'DP05_0009E',
 'DP05_0010E',
 'DP05_0011E',
 'DP05_0012E',
 'DP05_0013E',
 'DP05_0014E',
 'DP05_0015E',
 'DP05_0016E',
 'DP05_0017E',
 'DP05_0035E',
 'DP05_0045E',
 'DP05_0053E',
 'DP05_0061E',
 'DP05_0069E',
 'DP05_0074E',
 'DP05_0090E',
 'DP05_0096E',
 'B08301_003E',
 'B08301_004E',
 'B08301_010E',
 'B08301_016E',
 'B08301_017E',
 'B08301_018E',
 'B08301_019E',
 'B08301_020E',
 'B08301_021E',
 'B08006_020E',
 'B08006_021E',
 'B08006_025E',
 'B08006_031E',
 'B08006_032E',
 'B08006_033E',
 'B08006_034E',
 'B08006_037E',
 'B08006_038E',
 'B08006_042E',
 'B08006_048E',
 'B08006_049E',
 'B08006_050E',
 'B08006_051E',
 'B08101_010E',
 'B08101_011E',
 'B08101_012E',
 'B08101_013E',
 'B08101_014E',
 'B08101_015E',
 'B08101_016E',
 'B08101_018E',
 'B08101_019E',
 'B08101_020E',
 'B08101_021E',
 'B08101_022E',
 'B08101_023E',
 'B08101_024E',
 'B08101_026E',
 'B08101_027E',
 'B08101_028E',
 'B08101_029E',

# Scraping the JSONs to Collect the Variable Names for More Efficiently

In [16]:
year_start = 2011
year_end = 2025

field_name_df = pd.DataFrame()

table_type_extensions = {
    'detailed': '',
    'subject': '/subject',
    'profile': '/profile',
    'comparison': '/cprofile'
}     

def append_to_field_name_dict(year, table_type):
    url = f'https://api.census.gov/data/{year}/acs/acs5{table_type_extensions[table_type]}/variables.json'
    response = requests.get(url)
    
    try:
        fields_for_year_df = pd.DataFrame(response.json()['variables'].values())

        fields_for_year_df['Year'] = year
        fields_for_year_df['Name'] = response.json()['variables'].keys()
        
        global field_name_df
        field_name_df = pd.concat([field_name_df, fields_for_year_df])
        
    except:
        print(f'Error transcribing {table_type} table fields for {year}')
    time.sleep(5 + random.randint(-5000, 5000)/10000)
        
for year in range(year_start, year_end):
    append_to_field_name_dict(year, 'detailed')
    append_to_field_name_dict(year, 'subject')
    append_to_field_name_dict(year, 'profile')
    append_to_field_name_dict(year, 'comparison')
    
field_name_df['TableName'] = [field_name[0:field_name.index('_')] if '_' in field_name else None
                              for field_name in field_name_df.Name.values ]
field_name_df = field_name_df[['Year', 'Name', 'TableName'] + field_name_df.columns.tolist()[:-3]]
field_name_df = field_name_df.reset_index()

Error transcribing comparison table fields for 2011
Error transcribing comparison table fields for 2012
Error transcribing comparison table fields for 2013
Error transcribing comparison table fields for 2014


## Warning: Certain table concepts have different table names across the years
The following variable concepts_inconsistent_names contains a list of table concepts whose table names change at least once from 2011-2024. The code for collecting the field names for each year does not account for changes to table names, so special consideration needs to be taken if using these fields

In [265]:
# table_names_per_concept = field_name_df[~(field_name_df.TableName.isna())]\
#     .groupby(['TableName', 'concept'], as_index=False).count()\
#     [['TableName', 'concept']].groupby('concept', as_index=False).count()

# concepts_inconsistent_names = table_names_per_concept.loc[(table_names_per_concept.TableName > 1)]\
#     .sort_values('TableName', ascending=False).concept.values.tolist()

# Cleaning Up the Label Names

In [17]:
def standardize_label(label):
    standardized_label = label.replace('Total population!!', '')\
        .replace(':', '')\
        .replace('Total!!', '')\
        .replace('Taxicab', 'Taxi or ride-hailing services')\
        .replace('Public transportation (excluding taxicab)', 'Public transportation')\
        .replace('Public transportation:', 'Public transportation')\
        .replace('Worked from home', 'Worked at home')\
        .lower()
    return(standardized_label)

In [18]:
field_names_2024 = demographic_fields + commuting_fields + \
    commuting_by_age_fields + commuting_by_race_fields
table_names_2024 = list(set([field_name[0:field_name.index('_')] for field_name in field_names_2024]))

field_name_df['StandardizedLabel'] = [standardize_label(label) for label in field_name_df.label.values]
field_labels_2024_df = field_name_df.loc[(field_name_df.Name.isin(field_names_2024)) & 
                      (field_name_df.Year == 2024)][['StandardizedLabel', 'TableName', 'Name']]

In [19]:
field_labels_all = field_labels_2024_df[['TableName', 'StandardizedLabel']]\
    .merge(field_name_df.loc[(field_name_df.Year != 2024)][['Year', 'TableName', 'StandardizedLabel']], 
           on=['TableName', 'StandardizedLabel'], 
           how='outer', indicator=True)

field_labels_all['LabelParts'] = [label.split('!!') for label in field_labels_all.StandardizedLabel]

field_labels_missing = field_labels_all.loc[(field_labels_all._merge == 'left_only')]\
    [['TableName', 'StandardizedLabel']]
field_labels_remaining = field_labels_all.loc[(field_labels_all._merge == 'right_only')]\
    .loc[(field_labels_all.TableName).isin(field_labels_2024_df.TableName.values)]\
    [['TableName', 'StandardizedLabel']]

In [20]:
duplicate_field_labels = field_name_df[['TableName', 'StandardizedLabel', 'Name', 'Year']]\
    .merge(field_labels_2024_df.drop('Name', axis=1), on=['TableName', 'StandardizedLabel'])\
    .groupby(['Year', 'TableName', 'StandardizedLabel'], as_index=False).count()
duplicate_field_labels = duplicate_field_labels.loc[(duplicate_field_labels.Name) > 1]\
    .rename(columns={'Name': 'Count'})

In [21]:
duplicate_field_labels.head()

,Year,TableName,StandardizedLabel,Count
120,2011,DP05,estimate!!race!!two or more races,2
133,2011,DP05,estimate!!sex and age!!female,3
134,2011,DP05,estimate!!sex and age!!male,3
256,2012,DP05,estimate!!race!!two or more races,2
269,2012,DP05,estimate!!sex and age!!female,3


In [22]:
checking_duplicate_fields = duplicate_field_labels\
    .merge(field_name_df, on=['Year', 'TableName', 'StandardizedLabel'])\
    [['Year', 'TableName', 'StandardizedLabel', 'label', 'Name']]\
    .groupby(['Year', 'TableName', 'StandardizedLabel', 'label'], as_index=False).count()\
    .groupby(['Year', 'TableName', 'StandardizedLabel'], as_index=False).count()\
    .rename(columns={'label': 'Count'})

In [23]:
checking_duplicate_fields.loc[checking_duplicate_fields.Count > 1].head()

,Year,TableName,StandardizedLabel,Count,Name
6,2013,DP05,estimate!!race!!two or more races,2,2
7,2014,DP05,estimate!!race!!two or more races,2,2
8,2015,DP05,estimate!!race!!two or more races,2,2
9,2016,DP05,estimate!!race!!two or more races,2,2


In [24]:
duplicate_field_labels.merge(field_name_df, on=['Year', 'TableName', 'StandardizedLabel'])\
    [['Year', 'TableName', 'StandardizedLabel', 'Name', 'label']]

,Year,TableName,StandardizedLabel,Name,label
0,2011,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Two or more races
1,2011,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
2,2011,DP05,estimate!!sex and age!!female,DP05_0024E,Estimate!!SEX AND AGE!!Female
3,2011,DP05,estimate!!sex and age!!female,DP05_0027E,Estimate!!SEX AND AGE!!Female
4,2011,DP05,estimate!!sex and age!!female,DP05_0003E,Estimate!!SEX AND AGE!!Female
5,2011,DP05,estimate!!sex and age!!male,DP05_0026E,Estimate!!SEX AND AGE!!Male
6,2011,DP05,estimate!!sex and age!!male,DP05_0023E,Estimate!!SEX AND AGE!!Male
7,2011,DP05,estimate!!sex and age!!male,DP05_0002E,Estimate!!SEX AND AGE!!Male
8,2012,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Two or more races
9,2012,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races


Most of the fields with duplicated StandardizedLabels also have identical labels, so they should return the same data and we will only keep the first of these duplicate fields in each year.

For each of the years from 2013 to 2016, there are two rows in table DP05 that have similar labels. These labels don't have any apparent difference in meaning, and some quick testing indicates they consistently return the same data, so we will just keep one of the column names

In [25]:
duplicate_field_labels[(duplicate_field_labels.TableName == 'DP05') &
                      (duplicate_field_labels.Year.isin([2013, 2014, 2015, 2016]))]\
    .merge(field_name_df, on=['Year', 'TableName', 'StandardizedLabel'])\
    [['Year', 'TableName', 'StandardizedLabel', 'Name', 'label']]

,Year,TableName,StandardizedLabel,Name,label
0,2013,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
1,2013,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
2,2014,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
3,2014,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
4,2015,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
5,2015,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races
6,2016,DP05,estimate!!race!!two or more races,DP05_0030E,Estimate!!RACE!!Total population!!Two or more ...
7,2016,DP05,estimate!!race!!two or more races,DP05_0053E,Estimate!!RACE!!Two or more races


# Identifying the Necessary Columns for Each Year

In [29]:
field_names_by_year_df = field_labels_2024_df[['TableName', 'StandardizedLabel', 'Name']]\
    .rename(columns={'Name':'FieldName2024'})\
    .merge(field_name_df[['Year','TableName', 'StandardizedLabel', 'Name']],
          on=['TableName', 'StandardizedLabel'])\
    .sort_values('Name')\
    .groupby(['Year', 'TableName', 'StandardizedLabel'], as_index=False)\
    .first()

In [30]:
field_names_by_year_df.groupby(['Year', 'Name'], as_index=False).count()\
    .groupby(['Year']).count()

,Name,TableName,StandardizedLabel,FieldName2024
Year,,,,
2011,136,136,136,136
2012,136,136,136,136
2013,136,136,136,136
2014,136,136,136,136
2015,136,136,136,136
2016,136,136,136,136
2017,136,136,136,136
2018,136,136,136,136
2019,136,136,136,136


In [31]:
field_names_by_year_df.to_csv('Data/CensusDataFieldsToQuery.csv', index=False)